# Dataset

In [ ]:
# INPUTS for  Dataset & Trainer

# Trainer ------------------------------------------------------------------------------------------------------------------
objective = 'pred_x0' # 'pred_noise' , 'pred_x0', 'pred_v'

In [ ]:
# file path ----------------------------------------------------------------------------------------------------------
data_dir = ''    # where ECoG data folder is saved 
raw_dir  = ''    # where chan info is saved
experiment_dir = './experiments'
device_num = 0

sub_num = 0
train_session = 1

# target: / condition: -------------------------------------------------------
target = ''
condition = '' # ch_name like [37,31,6,15]

target_arr = []
cond_arr = []


rand_cond_flag = False

In [ ]:
# get data
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(device_num)

import torch
import numpy as np
import mne

import math
from scipy.integrate import cumtrapz

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import sys
import matplotlib

if sys.platform.startswith('win'):
    # Windows
    #matplotlib.use('Qt5Agg')
    %matplotlib qt
    print("Using qt backend")
else:
    # Linux / headless
    matplotlib.use('Agg')
    print("Using Agg backend")

import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
# load data

subs = {
    1:  {'name': '',    
         'bads': [], # bad channel name
         'depths': [], # non-ecog channel name
         'session': 0,  # session to use
         'drop_s1': [], # bad trial num
         'drop_s2': [], # bad trial num
         'notch': [], # notch filter (not used)
         'n_session1': 0}, # number of trials in session 1
}

sub_name = subs[sub_num]['name']
session_num = subs[sub_num]['session']

In [ ]:
def find_max_distance_timepoint(y, trig=100):
    # y shape: (3, seq_len, #trial)
    # trig   : timepoints of task trigger
    
    start_points = y[:, trig, :]
    subsequences = y[:, trig:, :]  # Shape: (3, seq_len - trig, #trial)
    distances = np.linalg.norm(subsequences - start_points[:, np.newaxis, :], axis=0)  # Shape: (seq_len-trig, #trial)
    max_distance_indices = np.argmax(distances, axis=0)
    reach_times = max_distance_indices + trig

    return reach_times


# load data
def load_data(sub_num, X_mode, y_mode, dsfreq=100):
    
    # set parameters -------------------------------------------------------------------------------------------------------------------------------------------
    rmv = np.array([], dtype=int)  # save bad trial idx
    accumulated_trial = 0 # count total trial
    
    # nsteps: for sequence dataset [-1,3]
    tmin, tmax = 1, 3
    
    # initialize
    # X data
    X_ME = np.zeros([0, 0, 0])
    X_MI = np.zeros([0, 0, 0])
    # y data
    y_vel = np.zeros([3, 0, 0])
    y_pos =  np.zeros([3, 0, 0])
    # sequence (4 direction label)
    sequences = np.zeros([0,])
        
    # 1. load all session data -----------------------------------------------------------------------------------------------------------------------------------
    for session in range(1, session_num + 1):
        
        # load data ---
        fname = f"{}.fif"
        epochs = mne.read_epochs(fname, preload=True)
        
        epochs_ME = epochs['ME']
        epochs_MI = epochs['MI']
        
        # load meta data ---
        fname = f"{}.npz"
        meta = np.load(fname)
        
        pos = meta['pos'] # (3, datapoint, trials)
        vel = meta['vel'] # (3, datapoint, trials)
        sequence = meta['sequence'] # (trial, 1)

        # load channel info ---
        fname = f'{}.pkl'
        with open(fname, 'rb') as f:
            ch_info = pickle.load(f)
    
        # get parameters
        sfreq = epochs.info["sfreq"] # 2000 Hz
        s_ratio = int(sfreq/dsfreq) 
    
        picks = mne.pick_types(epochs.info, meg=True, eeg=True, eog=True, stim=True, exclude='bads')
        nchan = len(picks)
    
        data_len = len(range(-np.int64(np.round(sfreq*tmin)), np.int64(np.round(sfreq*tmax)) + 1, s_ratio)) # downsampled to 100Hz
    
        # == TO REMOVE BAD TRIALS ==
        drop = f'drop_s{session}'
        drops =  subs[sub_num][drop]
    
        me_rmv = np.array([], dtype=int); mi_rmv = np.array([], dtype=int)
        for idx in drops: 
            n = idx // 2
            if idx % 2 == 0: me_rmv = np.append(me_rmv, n)    # ME              
            else: mi_rmv = np.append(mi_rmv, n)
    
        if X_mode == 'ME': rmv = np.append(rmv, me_rmv + accumulated_trial)
        else: rmv = np.append(rmv, mi_rmv + accumulated_trial)

        # save sequence
        sequences = np.append(sequences, sequence)
    
        # Make X, y data -----------------------------------------------------------------------------------------------------------------------------
    
        # initialize
        if X_ME.shape[-1] == 0:
            X_ME = np.zeros([nchan, data_len, 0]); X_MI = np.zeros([nchan, data_len, 0])
            y_vel = np.zeros([3, data_len, 0]);    y_pos =  np.zeros([3, data_len, 0])
           
        # Downsampling: make X data (epochs, chan, datapoints) --> (chan, datapoitns, epochs) 
        data_ME = epochs_ME.get_data(copy=True) # (epochs, channels, time points)
        seg_ME = data_ME[:, :, ::s_ratio]
        seg_ME = np.transpose(seg_ME, (1, 2, 0)) # (chan, datapoints, epochs)
        X_ME = np.append(X_ME, seg_ME, axis = 2)
    
        data_MI = epochs_MI.get_data(copy=True)
        seg_MI = data_MI[:, :, ::s_ratio]
        seg_MI = np.transpose(seg_MI, (1, 2, 0))
        X_MI = np.append(X_MI, seg_MI, axis = 2)
        
        # Downsampling: make y data
        seg_vel = vel[:, ::s_ratio, :]
        y_vel = np.append(y_vel, seg_vel, axis =2)
    
        seg_pos = pos[:, ::s_ratio, :]
        y_pos = np.append(y_pos, seg_pos, axis =2)
    
        print('\n >> %d Events' %(X_ME.shape[-1])) # session 1 / session 2 Events
        accumulated_trial += X_ME.shape[-1]
    
                   
    # 2. set X, y ----------------------------------------------------------------------------------------------------------------------------------
    
    # Set data (remove bad trial idx for y data)
    if X_mode == 'ME': X = X_ME
    elif X_mode == 'MI': X = X_MI
    
    if y_mode == 'pos': y = y_pos; label = y_pos
    elif y_mode == 'vel': y = y_vel; label = y_pos

    # find time for reaching task
    reach_times = find_max_distance_timepoint(label, trig=100)
        
    X = np.delete(X, rmv, axis=2)
    y = np.delete(y, rmv, axis=2)
    label = np.delete(label, rmv, axis=2)
    sequences = np.delete(sequences, rmv)       # 4 direction label
    reach_times = np.delete(reach_times, rmv)

    # FOR DIFFUSION ----------------------------------------------------
    X = np.transpose(X, (2, 0, 1)) # (trial, chan, datapoints)

    return X, y, label, sequences, reach_times, epochs, ch_info

In [ ]:
# load data ------------------------------------------------------------------------
X_mode = 'ME'  
y_mode = 'vel' 
dsfreq = 100  

X, _, _, _, _, epochs, ch_info = load_data(sub_num, X_mode, y_mode, dsfreq) 
# X: (#trial, chan, datapoints) // if not diffusion: (chan, datapoints, #trial)

### diffusion dataset

In [ ]:
from ecog_ddpm import Unet1D, GaussianDiffusion1D, Trainer1D, Dataset1D 

# make dataset for Diffusion Trainer

# get target, condition idx from channel name ------------------------------------------------------------------
total_ch_names = epochs.info.ch_names

def get_chan_idx(channels, total_ch_names = total_ch_names):
    indices = []
    for ch in channels:
        if str(ch) in total_ch_names:
            indices.append(total_ch_names.index(str(ch)))
        else:
            print(f"Warning: Chan {ch} not found in total_ch_names.")
    return indices

target_idx = get_chan_idx(target_chan)
cond_idx = get_chan_idx(condition_chan)

# normalize 0 to 1 for each chan ---------------------------------------------------------------------------------
def normalize_channel_data(data, channels, rand_flag=False, mode='zero_to_one'):
    normalized_data = []
    min_vals = []
    max_vals = []

    assert mode in {'zero_to_one', 'minus_one_to_one'}, 'mode must be either "zero_to_one" or "minus_one_to_one"'

    data = data.astype(np.float32)

    if data.shape[-1] % 2 == 1:
        data = data[:, :, :-1]
    
    # normalize each channel
    for chan in channels:

        channel_data = data[:, chan:chan+1, :].astype(np.float32)
    
        min_val = np.min(channel_data, axis=(0, 2), keepdims=False)
        max_val = np.max(channel_data, axis=(0, 2), keepdims=False)
        
        # normalization; min-max scaling
        if mode == 'zero_to_one':
            channel_data_normalized = (channel_data - min_val) / (max_val - min_val)
        elif mode == 'minus_one_to_one':
            channel_data_normalized = 2 * (channel_data - min_val) / (max_val - min_val) - 1
        else:
            raise ValueError(f'unknown mode {mode}')
        
        normalized_data.append(channel_data_normalized)
        
        # save min, max
        min_vals.append(min_val.item())
        max_vals.append(max_val.item())
    
    # concat all chan
    normalized_data = np.concatenate(normalized_data, axis=1)
    normalized_tensor = torch.from_numpy(normalized_data)

    if rand_flag:
        if mode == 'zero_to_one':
            normalized_tensor  = torch.rand_like(normalized_tensor)
            print("=== !Condition Data Randomized (0 to 1)! ===\n")
        elif mode == 'minus_one_to_one':
            normalized_tensor = 2 * torch.rand_like(normalized_tensor) - 1
            print("=== !Condition Data Randomized (-1 to 1)! ===\n")
        else:
            raise ValueError(f'unknown mode {mode}')
    
    return normalized_tensor, min_vals, max_vals



# Trainset ---------------------------------------------------------------------------------------

# train : using all data (for cross subjects)
if train_session == 0:

    # normalize
    target_data, target_min, target_max = normalize_channel_data(X, target_idx, False) 
    condition_data, condition_min, condition_max = normalize_channel_data(X, cond_idx, rand_cond_flag)
    
    train_trial, n_target_chan, seq_len = target_data.shape
    train_trial, n_condition_chan, seq_len = condition_data.shape
    trainset = Dataset1D(target_data, condition_data)

    # denormalize using train target
    denorm_target_min = target_min; denorm_target_max = target_max
        

# train: using 1 session data (for cross session / same subject)
else:

    # divide session 1 & 2
    n_session1 = subs[sub_num]['n_session1']
    X_s1 = X[:n_session1]
    X_s2 = X[n_session1:]
    
    # session 1
    target_s1_data, target_s1_min, target_s1_max = normalize_channel_data(X_s1, target_idx, False) # 무조건 false여야댐
    condition_s1_data, condition_s1_min, condition_s1_max = normalize_channel_data(X_s1, cond_idx, rand_cond_flag)
    
    # session 2
    target_s2_data, target_s2_min, target_s2_max = normalize_channel_data(X_s2, target_idx, False) # 무조건 false여야댐
    condition_s2_data, condition_s2_min, condition_s2_max = normalize_channel_data(X_s2, cond_idx, rand_cond_flag)
    
    if train_session == 1:
        
        train_trial, n_target_chan, seq_len = target_s1_data.shape
        train_trial, n_condition_chan, seq_len = condition_s1_data.shape     
        trainset = Dataset1D(target_s1_data, condition_s1_data)
        # denormalize using train target
        denorm_target_min = target_s1_min; denorm_target_max = target_s1_max
        
        test_trial, n_target_chan, seq_len = target_s2_data.shape
        test_trial, n_condition_chan, seq_len = condition_s2_data.shape
        testset = Dataset1D(target_s2_data, condition_s2_data)
    
    elif train_session == 2:
        
        train_trial, n_target_chan, seq_len = target_s2_data.shape
        train_trial, n_condition_chan, seq_len = condition_s2_data.shape     
        trainset = Dataset1D(target_s2_data, condition_s2_data)
        # denormalize using train target
        denorm_target_min = target_s2_min; denorm_target_max = target_s2_max
        
        test_trial, n_target_chan, seq_len = target_s1_data.shape
        test_trial, n_condition_chan, seq_len = condition_s1_data.shape
        testset = Dataset1D(target_s1_data, condition_s1_data)

    else:
        raise ValueError(f"Invalid train_session value: {train_session}. Must be either 0, 1, or 2.")

print(f'------------------- Train: session {train_session} --------------------')
print(f"Train trial: {train_trial}")
print(f"Trainset shape | x: {trainset.x.shape} || condition_x: {trainset.condition_x.shape}")

# Trainer

In [ ]:
from datetime import datetime
from pathlib import Path
import json

# set model path -------------------------------------------------------
date = datetime.now().strftime('%m%d')

target_str = []
cond_str = []

results_parent = f'{experiment_dir}/mdl'
log_parent = f'{experiment_dir}/log'

Path(results_parent).mkdir(exist_ok=True, parents=True)
Path(log_parent).mkdir(exist_ok=True, parents=True)

# save config file -----------------------------------------------------
Unet_param = {
    "dim": 32, 
    "dim_mults": (1,2,4,8),
    "target_dim": n_target_chan,  
    "condition_dim": n_condition_chan, 
    "cond_out_dim": n_condition_chan * 16, 
    "out_dim": n_target_chan 
}

diffusion_param = {
    "seq_length": seq_len,
    "timesteps": 100,
    "objective": objective
}


trainer_param = {
    "con_drop_prob": 0,
    "train_batch_size": 32,
    "train_lr": 8e-5,
    "train_num_steps": 20000,                     # total training steps
    "gradient_accumulate_every": 2,               # gradient accumulation steps
    "save_and_sample_every": 1000,                # sampling
    "num_samples": 2,                             # num samples to generate
    "ema_decay": 0.995,                           # exponential moving average decay
    "amp": True,                                  # turn on mixed precision
    "results_folder": results_parent,
    "log_dir": log_parent,
}

config = {
    "sub_num": sub_num,
    "target": target_chan,
    "condition": condition_chan,
    "train_session": train_session,
    "rand_cond_flag": rand_cond_flag,
    "denorm_target_min": denorm_target_min,
    "denorm_target_max": denorm_target_max,
    "Unet_param": Unet_param,
    "diffusion_param": diffusion_param,
    "trainer_param": trainer_param   
}

json_path = Path(results_parent) / 'config.json'
with json_path.open('w') as file:
    json.dump(config, file, indent=4)

# mdl init ----------------------------------------------------------------

with json_path.open("r") as file:
    config = json.load(file)


# Unet model ---
# [list] -> (tuple)
config["Unet_param"]["dim_mults"] = tuple(config["Unet_param"]["dim_mults"])
model = Unet1D(**config["Unet_param"])

# diffusion ---
diffusion = GaussianDiffusion1D(
    model,
    **config["diffusion_param"]
)

# Trainer ---
trainer = Trainer1D(
    diffusion,
    dataset = trainset,
    **config["trainer_param"]
)

### train

In [ ]:
# # load checkpoint ---------------
# milestone = '55'
# trainer.load(milestone)

# start training ------------------
trainer.train()